# Day 6 Capstone Mini-Challenge (Starter) — Complete Config-Driven ML Pipeline

**Timebox:** 30 minutes
**Challenge:** Put ALL 6 Mandatory + 3 Overview Day 6 skills together into ONE clean, repeatable, config-driven churn pipeline for StreamPulse:

1. **Config Management**: Read `configs/train_config.yaml` (no hardcoded paths or hyperparameters!).
2. **Data Ingestion**: Load & merge `customers_raw.csv` + `usage_events.json`, normalize strings, deduplicate `customer_id`.
3. **Data Validation**: Assert schema, uniqueness, allowed categories, and numeric bounds before training.
4. **Feature Engineering**: Add the 4 domain features specified in `configs/train_config.yaml`.
5. **Training & Testing Pipeline**: Build a leak-free `Pipeline`, run stratified K-fold CV, evaluate on held-out test set.
6. **Reproducibility & Experiment Tracking**: Fix random seed, save model artifact (`.joblib`), and append a structured run record to `artifacts/capstone_experiments.jsonl`.

Complete the `# TODO` sections below and run the final automated grader cell!


In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import random
import uuid

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Step 1: Load configuration from YAML
with open("configs/train_config.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

print("Loaded project:", CFG["project"]["name"])
print("Configured model:", CFG["model"]["type"], CFG["model"]["params"])


In [ ]:
# ============================================================================
# STEP 2: Data Ingestion
# ============================================================================
def stage_ingest(cfg: dict) -> tuple[pd.DataFrame, str]:
    """Load CSV + JSON from cfg paths, normalize strings, deduplicate, and merge.
    Returns (merged_df, customers_sha256).
    """
    # TODO 1: Load cfg["data"]["customers_csv"] and cfg["data"]["usage_json"],
    #         compute 12-char sha256 of the customers file, normalize plan_tier & country,
    #         drop duplicate customer_ids, and left-merge usage onto customers.
    raise NotImplementedError("Complete stage_ingest()")


In [ ]:
# ============================================================================
# STEP 3: Data Validation
# ============================================================================
def stage_validate(df: pd.DataFrame, cfg: dict) -> dict:
    """Validate df against cfg['validation'] rules. Raise ValueError if critical checks fail."""
    # TODO 2: Check:
    #   - customer_id is unique
    #   - non-null age within [min_age, max_age]
    #   - non-null tenure_months >= min_tenure_months
    #   - non-null plan_tier values subset of allowed_plan_tiers
    # Return {"passed": True, "rows": len(df)} (or raise ValueError on failure)
    raise NotImplementedError("Complete stage_validate()")


In [ ]:
# ============================================================================
# STEP 4: Feature Engineering
# ============================================================================
def stage_engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add charges_per_login, tickets_per_tenure_month, watch_hours_per_login, is_new_customer."""
    # TODO 3: Add the 4 domain features on a copy of df and return it
    raise NotImplementedError("Complete stage_engineer_features()")


In [ ]:
# ============================================================================
# STEP 5 & 6: Train/Test Pipeline + Artifact Saving + Experiment Logging
# ============================================================================
def stage_train_and_log(df: pd.DataFrame, data_hash: str, cfg: dict, log_path: Path) -> dict:
    """Run full config-driven training, evaluation, artifact save, and JSONL experiment log."""
    # TODO 4:
    #   - Set random seed from cfg["reproducibility"]["random_seed"]
    #   - Stratified train/test split using cfg["reproducibility"]["test_size"]
    #   - Build ColumnTransformer + estimator based on cfg["model"]["type"] and cfg["model"]["params"]
    #   - Compute CV ROC-AUC, fit on X_train, evaluate test ROC-AUC and test F1
    #   - Save fitted pipeline to artifacts/capstone_<run_id>.joblib
    #   - Append run record (run_id, model_type, data_hash, cv_roc_auc, test_roc_auc, test_f1, artifact_path)
    #     as a JSON line to log_path and return the record dict
    raise NotImplementedError("Complete stage_train_and_log()")


In [ ]:
# ============================================================================
# AUTOMATED CAPSTONE SELF-CHECK GRADER
# ============================================================================
capstone_log = Path("artifacts/capstone_experiments.jsonl")

raw_merged, data_hash = stage_ingest(CFG)
val_report = stage_validate(raw_merged, CFG)
featured_df = stage_engineer_features(raw_merged)
run_record = stage_train_and_log(featured_df, data_hash, CFG, capstone_log)

print("Capstone Run Record:")
print(json.dumps(run_record, indent=2))

assert val_report["passed"] is True
assert len(data_hash) == 12
assert run_record["cv_roc_auc"] >= 0.72
assert run_record["test_roc_auc"] >= 0.72
assert Path(run_record["artifact_path"]).exists(), "Saved model artifact missing!"
assert capstone_log.exists(), "Experiment log JSONL missing!"

# Smoke test reloading the saved model
reloaded = joblib.load(run_record["artifact_path"])
sample_X = featured_df[CFG["features"]["numeric_columns"] + CFG["features"]["categorical_columns"]].head(3)
sample_probs = reloaded.predict_proba(sample_X)[:, 1]
assert len(sample_probs) == 3 and np.all((sample_probs >= 0) & (sample_probs <= 1))

print("\n[CAPSTONE COMPLETE!] You just built a production-style, config-driven ML pipeline!")
